# Bike Availability vs Temperature Visualisation

# Importing all the Necessary Libraries

In [193]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# Loading the Datasets

In [194]:
bike_File = "/content/Dublin_Bikes_Station_Status_2025.csv"
weather_File = "/content/Dublin_Weather_Dataset_2025.csv"

In [195]:
def load_bikes_data(filepath):
    try:

        df = pd.read_csv(filepath, engine="python")
    except Exception as e:
        df = pd.read_csv(filepath, skiprows=[452659], engine="python")
    print(f"Bikes Data Loaded. Initial shape: {df.shape}")
    return df
bikes_df = load_bikes_data(bike_File)

Bikes Data Loaded. Initial shape: (639029, 15)


In [196]:
def load_weather_data(filepath):
    try:

        df = pd.read_csv(filepath, skiprows=23)
    except Exception:
        df = pd.read_csv(filepath, skiprows=24)
    print(f"Weather Data Loaded. Initial shape: {df.shape}")
    return df


weather_df = load_weather_data(weather_File)


Weather Data Loaded. Initial shape: (7297, 21)


# Initial Exploaration and EDA Process

In [197]:
print("Bikes Data Head and Information\n")
print(bikes_df.head())


Bikes Data Head and Information

      system_id        last_reported  station_id  num_bikes_available  \
0  dublin_bikes  2025-08-01 00:05:00           1                   13   
1  dublin_bikes  2025-08-01 00:05:00         100                   22   
2  dublin_bikes  2025-08-01 00:05:00         101                   13   
3  dublin_bikes  2025-08-01 00:05:00         103                   15   
4  dublin_bikes  2025-08-01 00:05:00         104                    9   

   num_docks_available  is_installed  is_renting  is_returning  \
0                   18          True        True          True   
1                    3          True        True          True   
2                   17          True        True          True   
3                   25          True        True          True   
4                   31          True        True          True   

                           name  short_name                       address  \
0                 CLARENDON ROW         NaN           

In [198]:
print("Weather Data Head and Information\n")
print(weather_df.head())


Weather Data Head and Information

               date  ind  rain  ind.1  temp  ind.2  wetb  dewpt  vappr  rhum  \
0  01-01-2025 00:00    0   0.8      0   6.7      0   6.0    5.2    8.8    90   
1  01-01-2025 01:00    0   1.2      0   6.1      0   5.5    4.7    8.5    91   
2  01-01-2025 02:00    0   1.2      0   5.3      0   4.8    4.2    8.2    92   
3  01-01-2025 03:00    0   1.3      0   5.3      0   4.8    4.1    8.2    92   
4  01-01-2025 04:00    0   0.6      0   5.0      0   4.6    4.1    8.2    94   

   ...  ind.3  wdsp  ind.4  wddir  ww   w  sun    vis  clht  clamt  
0  ...      2     9      2    250  60  62  0.0  12000    11      7  
1  ...      2    11      2    260  60  62  0.0   8000    14      7  
2  ...      2    14      2    260  61  66  0.0   8000    14      7  
3  ...      2    12      2    260  61  62  0.0   8000    13      7  
4  ...      2     9      2    240  60  62  0.0  16000    13      7  

[5 rows x 21 columns]


# Checking the Missig Values

In [199]:
print("Missing Values in Bikes Data\n")
print(bikes_df.isnull().sum())


Missing Values in Bikes Data

system_id                   0
last_reported               0
station_id                  0
num_bikes_available         0
num_docks_available         0
is_installed                0
is_renting                  0
is_returning                0
name                        0
short_name             639029
address                     0
lat                         0
lon                         0
region_id              639029
capacity                    0
dtype: int64


In [200]:
print("Descriptive Statistics for Key Bike Metrics\n")
print(bikes_df[["num_bikes_available", "num_docks_available", "capacity"]].describe().T)


Descriptive Statistics for Key Bike Metrics

                        count       mean        std   min   25%   50%   75%  \
num_bikes_available  639029.0  11.900717   9.778827   0.0   3.0  10.0  18.0   
num_docks_available  639029.0  19.615415  11.294731   0.0  11.0  19.0  29.0   
capacity             639029.0  31.840741   7.480782  16.0  29.0  30.0  40.0   

                      max  
num_bikes_available  40.0  
num_docks_available  40.0  
capacity             40.0  


In [201]:
# Rename for clarity during EDA

print("Descriptive Statistics for Temperature\n")
weather_df = weather_df.rename(columns={"temp": "Air Temperature (°C)"})
print(weather_df["Air Temperature (°C)"].describe())

Descriptive Statistics for Temperature

count    7297.000000
mean       11.223325
std         5.223827
min        -5.300000
25%         7.400000
50%        11.600000
75%        15.000000
max        27.600000
Name: Air Temperature (°C), dtype: float64


# Data Cleaning and Transformation

In [202]:
# Weather Data Cleaning

# Convert 'date' to datetime and create an hourly key for merging.

weather_df["date"] = pd.to_datetime(weather_df["date"], dayfirst=True)
weather_df["hour_key"] = weather_df["date"].dt.floor("h")
weather_df = weather_df[["hour_key", "Air Temperature (°C)"]]
print(f"Weather data ready for merge. Columns: {weather_df.columns.tolist()}")

Weather data ready for merge. Columns: ['hour_key', 'Air Temperature (°C)']


In [203]:
# Convert 'last_reported' to datetime and create the same hourly key.
bikes_df["last_reported"] = pd.to_datetime(bikes_df["last_reported"])
bikes_df["hour_key"] = bikes_df["last_reported"].dt.floor("h")

In [204]:
# Filter out rows where capacity is zero to avoid division-by-zero errors in metrics.

bikes_df = bikes_df[bikes_df["capacity"] > 0].copy()
print(f"Bikes data ready for merge. Columns: {bikes_df.columns.tolist()}")

Bikes data ready for merge. Columns: ['system_id', 'last_reported', 'station_id', 'num_bikes_available', 'num_docks_available', 'is_installed', 'is_renting', 'is_returning', 'name', 'short_name', 'address', 'lat', 'lon', 'region_id', 'capacity', 'hour_key']


In [205]:
# Merging Datasets
# Inner merge ensures we only analyze bike status reports that have corresponding weather data.
merged_df = pd.merge(
    bikes_df[["name", "num_bikes_available", "capacity", "num_docks_available", "hour_key"]],
    weather_df,
    on="hour_key",
    how="inner"
)
print(" Data Merged. Combined shape: {merged_df.shape}")


 Data Merged. Combined shape: {merged_df.shape}


# Metric Calculations

In [206]:
# Metric Set 1: Availability and Temperature per Station

# Group by station name and calculate the average availability and average temperature experienced.

summary_df = merged_df.groupby("name").agg(
    Avg_Bikes=("num_bikes_available", "mean"),
    Avg_Temp=("Air Temperature (°C)", "mean")
).reset_index()

# Define the dataset for the primary chart (Top 20 stations by availability)

top15_df = summary_df.sort_values("Avg_Bikes", ascending=False).head(20)

# Calculate a single reference line metric

overall_station_mean = top15_df["Avg_Bikes"].mean()
print(f"Overall average bikes available across top 20 stations: {overall_station_mean:.2f}")


# Metric Set 2: Station Capacity Composition
capacity_summary = merged_df.groupby("name").agg(
    Capacity=("capacity", "max"),
    Avg_Bikes_Available=("num_bikes_available", "mean"),
    Avg_Docks_Available=("num_docks_available", "mean")
).reset_index()

# Calculate average bikes "in use" (i.e., not available)

capacity_summary["Avg_Bikes_In_Use"] = capacity_summary["Capacity"] - capacity_summary["Avg_Docks_Available"]

# Calculate usage and empty dock percentages

capacity_summary["Used (%)"] = (capacity_summary["Avg_Bikes_In_Use"] / capacity_summary["Capacity"]) * 100
capacity_summary["Empty (%)"] = (capacity_summary["Avg_Docks_Available"] / capacity_summary["Capacity"]) * 100
capacity_summary["Available (%)"] = (capacity_summary["Avg_Bikes_Available"] / capacity_summary["Capacity"]) * 100

# Focus on the top 20 most utilized stations (highest Used % on average)

capacity_top20 = capacity_summary.sort_values("Used (%)", ascending=False).head(20).copy()
capacity_top20 = capacity_top20.rename(columns={"name": "Station Name"})

Overall average bikes available across top 20 stations: 20.64


In [207]:
# Bike Availability vs Temperature

# Bar chart showing the average number of bikes available for the Top 20 stations

# coloured by the average temperature recorded during the same periods.

print("Top 20 Bike Availability v/s Temperature in Dublin\n")
TEMPERATURE_SCALE = [
    [0.0, 'rgb(0, 110, 255)'],
    [0.5, 'rgb(255, 230, 0)'],
    [1.0, 'rgb(255, 0, 0)']
]

fig1 = px.bar(
    top15_df,
    x="Avg_Bikes",
    y="name",
    orientation="h",
    color="Avg_Temp",
    color_continuous_scale=TEMPERATURE_SCALE,
    title="<b>Top 20 Dublin Bike Availability in Station vs Temperature Visualisation</b>",
    template="plotly_white",
    height=750,
    labels={
        "Avg_Bikes": "Average Bikes Available",
        "Avg_Temp": "Average Temperature (°C)",
        "name": "Station Name"
    },
    custom_data=[top15_df["Avg_Temp"]]
)

# Improved hover text to show both metrics
fig1.update_traces(
    hovertemplate="<b>%{y}</b><br>Avg Bikes: %{x:.1f}<br>Temp: %{customdata[0]:.1f}°C<extra></extra>",
    marker_line_color="rgba(60,60,60,0.8)",
    marker_line_width=0.9
)

# Add a reference line for the overall average for comparison

fig1.add_vline(
    x=overall_station_mean,
    line_dash="dot",
    line_color="firebrick",
    line_width=2,
    annotation_text=f"Overall Top 20 Avg: {overall_station_mean:.1f}",
    annotation_position="top right",
    annotation_font_size=10
)

# Final layout adjustments

fig1.update_layout(
    title_font_size=18,
    title_x=0.5,
    font=dict(color="black"),
    xaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        zeroline=False,
        title="Average Number of Bikes Available"
    ),
    yaxis=dict(
        categoryorder="total ascending",
        title="Station Names"
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=160, r=80, t=80, b=80),
    coloraxis_colorbar=dict(title="Avg. Temp (°C)")
)

fig1.show()

Top 20 Bike Availability v/s Temperature in Dublin

